In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# KI-Transpiler-Passes
Die KI-gestützten Transpiler-Passes sind Passes, die als direkter Ersatz für „traditionelle" Qiskit-Passes bei bestimmten Transpilierungsaufgaben dienen. Sie liefern oft bessere Ergebnisse als bestehende heuristische Algorithmen (z. B. geringere Tiefe und CNOT-Anzahl) und sind zugleich deutlich schneller als Optimierungsalgorithmen wie SAT-Solver. Die KI-Transpiler-Passes können in deiner lokalen Umgebung oder in der Cloud über den Qiskit Transpiler Service ausgeführt werden, sofern du dem IBM Quantum&reg; Premium Plan, Flex Plan oder On-Prem (via IBM Quantum Platform API) Plan angehörst.

> **Note:** Die KI-gestützten Transpiler-Passes befinden sich im Beta-Status und können sich noch ändern.
>     Wenn du Feedback hast oder das Entwicklerteam kontaktieren möchtest, nutze diesen [Qiskit-Slack-Workspace-Kanal](https://qiskit.slack.com/archives/C06KF8YHUAU).

Folgende Passes sind derzeit verfügbar:

**Routing-Passes**
 - `AIRouting`: Layout-Auswahl und Circuit-Routing

**Circuit-Synthese-Passes**
 - `AICliffordSynthesis`: Clifford-Circuit-Synthese
 - `AILinearFunctionSynthesis`: Synthese linearer Funktions-Circuits
 - `AIPermutationSynthesis`: Permutations-Circuit-Synthese
 - `AIPauliNetworkSynthesis`: Pauli-Network-Circuit-Synthese

Um die KI-Transpiler-Passes zu verwenden, [installiere zunächst das Paket `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). In der [API-Dokumentation von qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) findest du weitere Informationen zu den verschiedenen verfügbaren Optionen.

## KI-Transpiler-Passes lokal oder in der Cloud ausführen
Wenn du die KI-gestützten Transpiler-Passes kostenlos in deiner lokalen Umgebung nutzen möchtest, installiere `qiskit-ibm-transpiler` mit einigen zusätzlichen Abhängigkeiten:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Ohne diese zusätzlichen Abhängigkeiten laufen die KI-gestützten Transpiler-Passes in der Cloud über den Qiskit Transpiler Service (nur verfügbar für Nutzer des IBM Quantum Premium Plan, Flex Plan oder On-Prem (via IBM Quantum Platform API) Plan). Nach der Installation der zusätzlichen Abhängigkeiten ist der Standardmodus für die KI-gestützten Transpiler-Passes der lokale Betrieb auf deinem Rechner.

## KI-Routing-Pass
Der `AIRouting`-Pass fungiert sowohl als Layout-Stage als auch als Routing-Stage. Er kann innerhalb eines `PassManager` wie folgt verwendet werden:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Hier legt `backend` fest, für welche Coupling Map geroutet wird. `optimization_level` (1, 2 oder 3) bestimmt den Rechenaufwand (höhere Werte liefern in der Regel bessere Ergebnisse, benötigen aber mehr Zeit), und `layout_mode` gibt an, wie die Layout-Auswahl gehandhabt wird.
`layout_mode` bietet folgende Optionen:

- `keep`: Respektiert das Layout, das von vorherigen Transpiler-Passes gesetzt wurde (oder verwendet das triviale Layout, falls keines gesetzt wurde). Wird typischerweise nur verwendet, wenn der Circuit auf bestimmten Qubits des Geräts ausgeführt werden muss. Liefert oft schlechtere Ergebnisse, da weniger Optimierungsspielraum besteht.
- `improve`: Verwendet das von vorherigen Transpiler-Passes gesetzte Layout als Ausgangspunkt. Nützlich, wenn du eine gute anfängliche Schätzung für das Layout hast – zum Beispiel bei Circuits, die in etwa der Coupling Map des Geräts folgen. Hilfreich auch, wenn du andere spezifische Layout-Passes in Kombination mit dem `AIRouting`-Pass ausprobieren möchtest.
- `optimize`: Dies ist der Standardmodus. Er eignet sich am besten für allgemeine Circuits, bei denen keine guten Layout-Schätzungen vorliegen. Dieser Modus ignoriert vorherige Layout-Auswahlen.
- `local_mode`: Dieses Flag bestimmt, wo der `AIRouting`-Pass ausgeführt wird. Bei `False` läuft `AIRouting` remote über den Qiskit Transpiler Service. Bei `True` versucht das Paket, den Pass in deiner lokalen Umgebung auszuführen, mit einem Fallback auf den Cloud-Modus, falls die erforderlichen Abhängigkeiten nicht gefunden werden.

## KI-Circuit-Synthese-Passes
Die KI-Circuit-Synthese-Passes ermöglichen es dir, Teile verschiedener Circuit-Typen ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) durch Re-Synthese zu optimieren. Eine typische Verwendung des Synthese-Passes sieht so aus:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Die Synthese berücksichtigt die Coupling Map des Geräts: Sie kann sicher nach anderen Routing-Passes ausgeführt werden, ohne den Circuit zu beeinträchtigen, sodass der gesamte Circuit weiterhin den Gerätebeschränkungen entspricht. Standardmäßig ersetzt die Synthese den ursprünglichen Sub-Circuit nur, wenn der synthetisierte Sub-Circuit eine Verbesserung darstellt (derzeit wird nur die CNOT-Anzahl geprüft). Durch Setzen von `replace_only_if_better=False` kann die Ersetzung jedoch immer erzwungen werden.

Folgende Synthese-Passes sind aus `qiskit_ibm_transpiler.ai.synthesis` verfügbar:

- *AICliffordSynthesis*: Synthese für [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford)-Circuits (Blöcke aus `H`-, `S`- und `CX`-Gates). Derzeit bis zu neun Qubit-Blöcke.
- *AILinearFunctionSynthesis*: Synthese für [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction)-Circuits (Blöcke aus `CX`- und `SWAP`-Gates). Derzeit bis zu neun Qubit-Blöcke.
- *AIPermutationSynthesis*: Synthese für [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation)-Circuits (Blöcke aus `SWAP`-Gates). Derzeit verfügbar für 65-, 33- und 27-Qubit-Blöcke.
- *AIPauliNetworkSynthesis*: Synthese für Pauli-Network-Circuits (Blöcke aus `H`-, `S`-, `SX`-, `CX`-, `RX`-, `RY`- und `RZ`-Gates). Derzeit bis zu sechs Qubit-Blöcke.

Wir planen, die unterstützte Blockgröße schrittweise zu erhöhen.

Alle Passes verwenden einen Thread-Pool, um mehrere Anfragen parallel zu senden. Standardmäßig entspricht die maximale Thread-Anzahl der Anzahl der Kerne plus vier (Standardwerte des Python-Objekts `ThreadPoolExecutor`). Du kannst jedoch beim Instanziieren des Passes über das Argument `max_threads` einen eigenen Wert festlegen. Die folgende Zeile instanziiert beispielsweise den `AILinearFunctionSynthesis`-Pass und erlaubt ihm, maximal 20 Threads zu verwenden.